# Week 3 &mdash; Message Passing and Spatial GNNs

## Theory: The MPNN Framework, Permutation Symmetry, GCN, and GAT

**Geometric Deep Learning &mdash; From Foundations to Applications**

---

### Learning Objectives

1. State the **Message Passing Neural Network (MPNN)** framework and its three primitive operations.
2. Prove that MPNNs are **permutation-equivariant** by construction.
3. Derive the **Graph Convolutional Network (GCN)** propagation rule and interpret its renormalisation.
4. Derive the **Graph Attention Network (GAT)** and contrast learned vs. fixed aggregation weights.
5. Understand the expressive limits of message passing via the **Weisfeiler&ndash;Lehman** test.

---


## 1. From Spectral to Spatial

Week 2 ended with the observation that polynomial spectral filters are *local*: they aggregate information within a $K$-hop neighbourhood. The **spatial** view takes this locality as the starting point and discards the spectral machinery entirely. Instead of diagonalising the Laplacian, we describe computation directly in terms of vertices exchanging *messages* along edges.

This perspective has decisive advantages: it requires no eigendecomposition, applies to graphs of arbitrary and varying size, handles edge and node features naturally, and is manifestly local. Almost every modern GNN &mdash; GCN, GraphSAGE, GAT, GIN &mdash; is a special case of the spatial framework introduced by Gilmer et al. (2017).


## 2. The Message Passing Neural Network Framework

Let $G = (V, E)$ carry node features $h_i^{(0)} = x_i \in \mathbb{R}^d$ and optional edge features $e_{ij}$. A single **message-passing layer** updates every node by three operations.

**(i) Message.** Each neighbour $j \in \mathcal{N}(i)$ computes a message to $i$:
$$
m_{ij}^{(l)} = \psi^{(l)}\!\big(h_i^{(l)},\, h_j^{(l)},\, e_{ij}\big).
$$

**(ii) Aggregate.** Node $i$ combines incoming messages with a **permutation-invariant** operator $\bigoplus$ (sum, mean, or max):
$$
a_i^{(l)} = \bigoplus_{j \in \mathcal{N}(i)} m_{ij}^{(l)}.
$$

**(iii) Update.** Node $i$ updates its state using its old state and the aggregate:
$$
h_i^{(l+1)} = \phi^{(l)}\!\big(h_i^{(l)},\, a_i^{(l)}\big).
$$

Here $\psi^{(l)}$ (message function) and $\phi^{(l)}$ (update function) are learnable, typically small MLPs. After $L$ layers, each node's representation depends on its $L$-hop receptive field. A final **read-out** $\bigoplus_{i \in V} h_i^{(L)}$ produces a permutation-invariant graph-level embedding when graph-level prediction is required.


## 3. Permutation Symmetry &mdash; The Defining Constraint

The symmetry group of an (unordered) graph is the permutation group $S_n$ acting simultaneously on node features and the adjacency matrix:
$$
X \mapsto PX, \qquad A \mapsto PAP^\top, \qquad P \text{ a permutation matrix.}
$$

**Proposition.** *An MPNN layer with a permutation-invariant aggregator $\bigoplus$ is permutation-equivariant.*

*Sketch.* The message $m_{ij}$ depends only on the *features* at the endpoints, not on node indices. The aggregate $\bigoplus_{j \in \mathcal{N}(i)} m_{ij}$ is, by assumption, invariant to the order in which neighbours are presented &mdash; reindexing the neighbours of $i$ leaves $a_i$ unchanged. The update applies the same $\phi$ to every node. Hence relabelling the graph relabels the outputs identically: $f(PX, PAP^\top) = P\, f(X, A)$. $\;\blacksquare$

The requirement that $\bigoplus$ be *order-invariant* is therefore not a stylistic choice but the precise condition that encodes the graph's symmetry into the architecture &mdash; exactly the Week 1 blueprint instantiated on the permutation group.


## 4. The Graph Convolutional Network (GCN)

Kipf &amp; Welling (2017) derive the GCN as a first-order ($K=1$) simplification of the ChebNet filter from Week 2. The layer-wise propagation rule is
$$
H^{(l+1)} = \sigma\!\Big(\hat{A}\, H^{(l)} W^{(l)}\Big), \qquad
\hat{A} = \tilde{D}^{-1/2}\,\tilde{A}\,\tilde{D}^{-1/2},
$$
where $\tilde{A} = A + I$ adds **self-loops** and $\tilde{D}$ is the degree matrix of $\tilde{A}$.

**Interpretation as message passing.** Writing the rule per node,
$$
h_i^{(l+1)} = \sigma\!\Big(\sum_{j \in \mathcal{N}(i)\cup\{i\}} \frac{1}{\sqrt{\tilde{d}_i\,\tilde{d}_j}}\; W^{(l)} h_j^{(l)}\Big),
$$
we recognise: the message is a *linear projection* $W h_j$, the aggregator is a **weighted sum** with fixed symmetric coefficients $1/\sqrt{\tilde{d}_i \tilde{d}_j}$, and the update is the nonlinearity $\sigma$.

**Why the renormalisation?** The symmetric normalisation $\tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}$ keeps the spectral radius bounded (eigenvalues in roughly $[0, 2]$ become $[-1, 1]$ after the self-loop trick), preventing the repeated multiplication across layers from causing numerical explosion or vanishing &mdash; a stability concern central to deep GNNs.


## 5. The Graph Attention Network (GAT)

The GCN's aggregation weights are *fixed* by the graph's degrees. Veli&#269;kovi&#263; et al. (2018) make them **learnable and feature-dependent** via an attention mechanism.

For each edge $(i, j)$, an unnormalised attention score is computed,
$$
e_{ij} = \mathrm{LeakyReLU}\!\Big(\mathbf{a}^\top \big[\, W h_i \,\|\, W h_j \,\big]\Big),
$$
where $\|$ denotes concatenation and $\mathbf{a}$ is a learnable vector. These are normalised across each node's neighbourhood with a softmax,
$$
\alpha_{ij} = \frac{\exp(e_{ij})}{\sum_{k \in \mathcal{N}(i)} \exp(e_{ik})},
$$
and the node is updated by an attention-weighted sum:
$$
h_i^{(l+1)} = \sigma\!\Big(\sum_{j \in \mathcal{N}(i)} \alpha_{ij}\, W h_j\Big).
$$

In practice **multi-head attention** runs $K$ independent attention mechanisms in parallel and concatenates (or averages) them, stabilising training and enriching the representation. GAT remains permutation-equivariant because the softmax is computed over the *set* of neighbours, independent of their ordering.

**GCN vs. GAT.** GCN uses *structural* (degree-based) weights known a priori; GAT *learns* edge importances from features, adapting aggregation to the task at the cost of additional parameters and computation.


## 6. Expressive Power: The Weisfeiler&ndash;Lehman Bound

How powerful is message passing? Xu et al. (2019) showed that *any* MPNN is **at most as expressive** as the **1-dimensional Weisfeiler&ndash;Lehman (1-WL) graph isomorphism test** in distinguishing non-isomorphic graphs.

The 1-WL test iteratively refines node colours by hashing each node's colour together with the *multiset* of its neighbours' colours &mdash; structurally identical to message passing with an injective aggregator. Consequently:

- MPNNs **cannot distinguish** any pair of graphs that 1-WL deems equivalent (e.g. certain regular graphs).
- An MPNN **achieves** the 1-WL bound only if both its aggregator and update are *injective* on multisets. The **Graph Isomorphism Network (GIN)**,
$$
h_i^{(l+1)} = \mathrm{MLP}^{(l)}\!\Big((1 + \epsilon^{(l)})\, h_i^{(l)} + \sum_{j \in \mathcal{N}(i)} h_j^{(l)}\Big),
$$
uses a *sum* aggregator (injective on multisets) to provably reach this maximum.

This result frames an active research frontier: higher-order GNNs, subgraph methods, and positional encodings all aim to surpass the 1-WL ceiling.


## 7. Summary

- The **MPNN** framework &mdash; *message, aggregate, update* &mdash; is the unifying spatial template for GNNs.
- A **permutation-invariant aggregator** makes the layer permutation-equivariant, encoding the graph's symmetry.
- **GCN** is a degree-normalised linear MPNN derived from first-order spectral filtering.
- **GAT** replaces fixed weights with learned, feature-dependent **attention**.
- Message passing is bounded in expressivity by the **1-WL test**; **GIN** attains this bound with a sum aggregator.

### References

- Gilmer, J., Schoenholz, S. S., Riley, P. F., Vinyals, O., &amp; Dahl, G. E. (2017). *Neural Message Passing for Quantum Chemistry.* ICML.
- Kipf, T. N., &amp; Welling, M. (2017). *Semi-Supervised Classification with Graph Convolutional Networks.* ICLR.
- Veli&#269;kovi&#263;, P., et al. (2018). *Graph Attention Networks.* ICLR.
- Xu, K., Hu, W., Leskovec, J., &amp; Jegelka, S. (2019). *How Powerful are Graph Neural Networks?* ICLR.

### Exercises

1. Show explicitly that the GCN rule with $\tilde{A} = A + I$ corresponds to a $K=1$ Chebyshev filter under the renormalisation trick.
2. Prove that the GAT softmax-weighted aggregation is permutation-equivariant.
3. Construct two non-isomorphic graphs that 1-WL (and hence any MPNN) cannot distinguish.
